# RIS-Assisted Communication Link

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jman4162/metasurface-py/blob/main/notebooks/03_ris_link.ipynb)

This notebook simulates a narrowband SISO RIS-assisted wireless link:
1. Define TX, RX, and RIS geometry
2. Compute analytically optimal RIS phases
3. Compare optimal, quantized, and no-RIS configurations
4. Verify N² SNR scaling with array size

In [ ]:
# !pip install metasurface-py

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from metasurface_py.geometry import RectangularLattice
from metasurface_py.elements import PhaseOnlyCell, DiscretePhaseSpace
from metasurface_py.elements.states import ContinuousPhaseSpace
from metasurface_py.surfaces import Metasurface
from metasurface_py.channels import RISLink, free_space_path_loss_db
from metasurface_py.core.types import Position3D
from metasurface_py.plotting import set_publication_style

set_publication_style()

## Scenario Setup

A base station (TX) communicates with a user (RX) assisted by a RIS at the origin.

```
TX (0, 0, 20m)  ----  RIS (origin)  ----  RX (10m, 0, 1.5m)
```

In [ ]:
freq = 28e9
lam = 3e8 / freq
tx = Position3D(x=0.0, y=0.0, z=20.0)
rx = Position3D(x=10.0, y=0.0, z=1.5)
tx_power_dbm = 20.0
noise_dbm = -90.0

# Direct link (no RIS)
d_direct = tx.distance_to(rx)
pl_direct = free_space_path_loss_db(d_direct, freq)
snr_direct = tx_power_dbm - pl_direct - noise_dbm
print(f"Direct link: distance = {d_direct:.1f} m, FSPL = {pl_direct:.1f} dB, SNR = {snr_direct:.1f} dB")

## SNR vs Array Size

For optimal continuous phases, the RIS power gain scales as N² (number of elements squared),
because all N elements contribute coherently.

In [ ]:
sizes = [8, 12, 16, 20, 24, 32]
snr_optimal = []
snr_2bit = []

for n in sizes:
    lattice = RectangularLattice(nx=n, ny=n, dx=lam/2, dy=lam/2)
    
    # Optimal continuous
    cell = PhaseOnlyCell(state_space=ContinuousPhaseSpace())
    surface = Metasurface(lattice=lattice, cell=cell)
    link = RISLink(surface=surface, tx=tx, rx=rx, freq=freq, include_direct=True)
    opt_state = link.optimal_state_continuous()
    snr_optimal.append(link.snr_db(opt_state, tx_power_dbm, noise_dbm))
    
    # 2-bit quantized
    cell_2b = PhaseOnlyCell(state_space=DiscretePhaseSpace(num_bits=2))
    surface_2b = Metasurface(lattice=lattice, cell=cell_2b)
    link_2b = RISLink(surface=surface_2b, tx=tx, rx=rx, freq=freq, include_direct=True)
    state_2b = link_2b.optimal_state_continuous().quantize(cell_2b.state_space.codebook)
    snr_2bit.append(link_2b.snr_db(state_2b, tx_power_dbm, noise_dbm))

n_elements = [n**2 for n in sizes]

In [ ]:
fig, ax = plt.subplots()
ax.plot(n_elements, snr_optimal, 'o-', label='Optimal (continuous)')
ax.plot(n_elements, snr_2bit, 's--', label='2-bit quantized')
ax.axhline(y=snr_direct, color='gray', linestyle=':', label='No RIS (direct)')
ax.set_xlabel('Number of RIS Elements')
ax.set_ylabel('SNR [dB]')
ax.set_title('RIS-Assisted Link — SNR vs Array Size')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print(f"\nSNR improvement (32×32 optimal vs no-RIS): {snr_optimal[-1] - snr_direct:.1f} dB")